# Fine-tune PhoBERT v2 for Fake News Detection

Load stratified train/val data, fine-tune PhoBERT v2, save full model

## 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import torch
import os
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("✅ Libraries imported successfully")
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")

✅ Libraries imported successfully
PyTorch version: 2.6.0+cu124
GPU available: True


## 2. Load Data

In [3]:
# Load train and validation sets
train_path = '../../data/splited/train_set.csv'
val_path = '../../data/splited/val_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

print("="*80)
print("DATA LOADED")
print("="*80)
print(f"\nTrain set: {df_train.shape[0]} rows × {df_train.shape[1]} columns")
print(f"  Label 0 (Fake): {(df_train['label'] == 0).sum()}")
print(f"  Label 1 (Real): {(df_train['label'] == 1).sum()}")

print(f"\nVal set: {df_val.shape[0]} rows × {df_val.shape[1]} columns")
print(f"  Label 0 (Fake): {(df_val['label'] == 0).sum()}")
print(f"  Label 1 (Real): {(df_val['label'] == 1).sum()}")

print(f"\nColumns: {list(df_train.columns)}")

DATA LOADED

Train set: 3788 rows × 28 columns
  Label 0 (Fake): 3143
  Label 1 (Real): 645

Val set: 474 rows × 28 columns
  Label 0 (Fake): 393
  Label 1 (Real): 81

Columns: ['id', 'post_message', 'label', 'num_char', 'num_emoji', 'num_url', 'num_hashtag', 'num_like', 'num_cmt', 'num_share', 'text_strict', 'text_loose', 'feat_word_count', 'feat_exclamation', 'feat_question', 'feat_uppercase_words', 'feat_uppercase_ratio', 'feat_punctuation_ratio', 'feat_engagement_total', 'feat_like_ratio', 'feat_comment_ratio', 'feat_like_per_char', 'feat_hour_sin', 'feat_hour_cos', 'feat_day_sin', 'feat_day_cos', 'feat_month_sin', 'feat_month_cos']


## 3. Setup Tokenizer

In [4]:
# Load PhoBERT v2 tokenizer
model_name = 'vinai/phobert-base-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("="*80)
print("TOKENIZER SETUP")
print("="*80)
print(f"\nModel: {model_name}")
print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Max length: 256")

# Test tokenizer
test_text = "Đây là một bài báo"
tokens = tokenizer(test_text, truncation=True, max_length=256)
print(f"\n✅ Tokenizer ready")
print(f"Sample tokens ('{test_text}'): {tokens['input_ids'][:10]}")

TOKENIZER SETUP

Model: vinai/phobert-base-v2
Vocab size: 64000
Max length: 256

✅ Tokenizer ready
Sample tokens ('Đây là một bài báo'): [0, 282, 8, 16, 387, 441, 2]


## 4. Prepare Datasets

In [6]:
def tokenize_function(examples):
    """Tokenize text data - handle edge cases"""
    # Ensure all texts are strings and non-empty
    texts = []
    for text in examples['text_loose']:
        # Convert to string, handle NaN
        text_str = str(text).strip() if pd.notna(text) else ""
        # Skip empty text
        if len(text_str) == 0:
            text_str = "[UNK]"  # Use unknown token for empty text
        texts.append(text_str)
    
    return tokenizer(
        texts,
        truncation=True,
        max_length=256,
        padding='max_length'
    )

# Convert to HuggingFace datasets
train_dataset = Dataset.from_pandas(df_train[['text_loose', 'label']])
val_dataset = Dataset.from_pandas(df_val[['text_loose', 'label']])

print("="*80)
print("TOKENIZING DATASETS")
print("="*80)
print(f"\nTokenizing train set...")
train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=32)

print(f"Tokenizing val set...")
val_dataset = val_dataset.map(tokenize_function, batched=True, batch_size=32)

# Set format for trainer
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"\n✅ Train dataset: {len(train_dataset)} samples")
print(f"✅ Val dataset: {len(val_dataset)} samples")

TOKENIZING DATASETS

Tokenizing train set...


TOKENIZING DATASETS

Tokenizing train set...


Map: 100%|██████████| 3788/3788 [00:02<00:00, 1596.34 examples/s]


TOKENIZING DATASETS

Tokenizing train set...


Map: 100%|██████████| 3788/3788 [00:02<00:00, 1596.34 examples/s]


...

TOKENIZING DATASETS

Tokenizing train set...


Map: 100%|██████████| 3788/3788 [00:02<00:00, 1596.34 examples/s]


Tokenizing val set...


Map: 100%|██████████| 474/474 [00:00<00:00, 1336.40 examples/s]


✅ Train dataset: 3788 samples
✅ Val dataset: 474 samples


## 5. Load Pre-trained Model

In [7]:
# Load PhoBERT v2 for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2  # 2 classes: fake (0) and real (1)
)

print("="*80)
print("MODEL LOADED")
print("="*80)
print(f"\nModel: {model_name}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Number of trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"\nDevice: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


MODEL LOADED

Model: vinai/phobert-base-v2
Number of parameters: 134,999,810
Number of trainable parameters: 134,999,810

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
GPU Memory: 4.29 GB


## 6. Define Metrics

In [8]:
def compute_metrics(eval_pred):
    """Compute evaluation metrics"""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions, average='weighted'),
        'recall': recall_score(labels, predictions, average='weighted'),
        'f1': f1_score(labels, predictions, average='weighted')
    }

print("✅ Metrics function defined")

✅ Metrics function defined


## 7. Fine-tune Model

In [9]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    learning_rate=2e-5,
    seed=42
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=None
)

print("="*80)
print("FINE-TUNING")
print("="*80)
print(f"\nTraining arguments:")
print(f"  Epochs: 3")
print(f"  Batch size: 16")
print(f"  Learning rate: 2e-5")
print(f"  Warmup steps: 500")

# Start training
train_result = trainer.train()

print(f"\n✅ Training completed")
print(f"Final loss: {train_result.training_loss:.4f}")

FINE-TUNING

Training arguments:
  Epochs: 3
  Batch size: 16
  Learning rate: 2e-5
  Warmup steps: 500


 14%|█▍        | 100/711 [03:21<20:34,  2.02s/it]

FINE-TUNING

Training arguments:
  Epochs: 3
  Batch size: 16
  Learning rate: 2e-5
  Warmup steps: 500


 14%|█▍        | 100/711 [03:21<20:34,  2.02s/it]

{'loss': 0.5917, 'grad_norm': 1.279013991355896, 'learning_rate': 4.000000000000001e-06, 'epoch': 0.42}


...

{'loss': 0.204, 'grad_norm': 7.26275634765625, 'learning_rate': 1.042654028436019e-06, 'epoch': 2.95}


                                                 
100%|██████████| 711/711 [24:13<00:00,  1.77s/it]

{'eval_loss': 0.30875465273857117, 'eval_accuracy': 0.9029535864978903, 'eval_precision': 0.8996971763412692, 'eval_recall': 0.9029535864978903, 'eval_f1': 0.9009217992414752, 'eval_runtime': 6.3211, 'eval_samples_per_second': 74.987, 'eval_steps_per_second': 4.746, 'epoch': 3.0}


100%|██████████| 711/711 [24:15<00:00,  2.05s/it]

{'train_runtime': 1455.8066, 'train_samples_per_second': 7.806, 'train_steps_per_second': 0.488, 'train_loss': 0.3104564928974951, 'epoch': 3.0}

✅ Training completed
Final loss: 0.3105


## 8. Evaluate on Validation Set

In [10]:
# Evaluate on validation set
eval_result = trainer.evaluate()

print("="*80)
print("VALIDATION RESULTS")
print("="*80)
print(f"\nAccuracy:  {eval_result['eval_accuracy']:.4f}")
print(f"Precision: {eval_result['eval_precision']:.4f}")
print(f"Recall:    {eval_result['eval_recall']:.4f}")
print(f"F1 Score:  {eval_result['eval_f1']:.4f}")
print(f"Loss:      {eval_result['eval_loss']:.4f}")

100%|██████████| 30/30 [00:06<00:00,  4.92it/s]

100%|██████████| 30/30 [00:06<00:00,  4.92it/s]

VALIDATION RESULTS

Accuracy:  0.9030
Precision: 0.8997
Recall:    0.9030
F1 Score:  0.9009
Loss:      0.3088


## 9. Save Model

In [13]:
# Create output directory
model_save_path = '../../model/finetuned/PhoBERT_v2'
os.makedirs(model_save_path, exist_ok=True)

# Save model (embeddings + encoders + classification head)
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print("="*80)
print("MODEL SAVED")
print("="*80)
print(f"\n✅ Full model saved to: {os.path.abspath(model_save_path)}")
print(f"\nModel contents:")
for file in os.listdir(model_save_path):
    file_path = os.path.join(model_save_path, file)
    if os.path.isfile(file_path):
        size_mb = os.path.getsize(file_path) / (1024**2)
        print(f"  - {file} ({size_mb:.1f} MB)")
    else:
        print(f"  - {file}/ (directory)")

total_size = sum(os.path.getsize(os.path.join(model_save_path, f)) 
                 for f in os.listdir(model_save_path) 
                 if os.path.isfile(os.path.join(model_save_path, f))) / (1024**2)
print(f"\nTotal size: {total_size:.1f} MB")
print(f"\n🚀 Ready for embedding extraction!")

MODEL SAVED

✅ Full model saved to: d:\Vietnamese-Fake-News-Detection\model\finetuned\PhoBERT_v2

Model contents:
  - added_tokens.json (0.0 MB)
  - bpe.codes (1.1 MB)
  - config.json (0.0 MB)
  - model.safetensors (515.0 MB)
  - special_tokens_map.json (0.0 MB)
  - tokenizer_config.json (0.0 MB)
  - vocab.txt (0.9 MB)

Total size: 516.9 MB

🚀 Ready for embedding extraction!


## 10. Test Prediction

In [14]:
# Test prediction on sample texts
sample_texts = [
    "Tin giả: Chính trị gia nổi tiếng bất ngờ qua đời",
    "Thông tin chính thức từ báo điện tử uy tín về sự kiện hôm nay"
]

print("="*80)
print("SAMPLE PREDICTIONS")
print("="*80)

for i, text in enumerate(sample_texts, 1):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=256, padding='max_length')
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
    
    fake_prob = probs[0][0].item()
    real_prob = probs[0][1].item()
    prediction = "FAKE" if fake_prob > real_prob else "REAL"
    
    print(f"\nText {i}: {text}")
    print(f"  Fake: {fake_prob:.4f} | Real: {real_prob:.4f}")
    print(f"  Prediction: {prediction}")

SAMPLE PREDICTIONS

Text 1: Tin giả: Chính trị gia nổi tiếng bất ngờ qua đời
  Fake: 0.9867 | Real: 0.0133
  Prediction: FAKE

Text 2: Thông tin chính thức từ báo điện tử uy tín về sự kiện hôm nay
  Fake: 0.9874 | Real: 0.0126
  Prediction: FAKE
